### import essential libraries

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

### Load Data

In [2]:
iris = load_iris()

In [3]:
X, y = iris['data'], iris['target']

### Preprocessing

In [4]:
# Standardize the data
scaler = StandardScaler()
X = scaler.fit_transform(X)

### train- test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Define imbalance ratios

In [6]:
imbalance_ratios = [(90, 5, 5), (85, 10, 5), (80, 15, 5), (75, 20, 5), (70, 20, 10)]

### Define a function to imbalance the data

In [7]:
def create_imbalanced_dataset(X, y, ratio):
    unique_classes = np.unique(y)
    if len(unique_classes) < 2:
        raise ValueError('The dataset must contain at least two classes.')

    X_imbalanced, y_imbalanced = [], []
    for i, cls in enumerate(unique_classes):
        samples = X[y == cls]
        n_samples = min(max(int(len(X) * ratio[i] / sum(ratio)), 1), len(samples))

        resampled_samples = resample(samples, replace=False, n_samples=n_samples, random_state=42)

        X_imbalanced.append(resampled_samples)
        y_imbalanced.extend([cls] * n_samples)

    X_imbalanced = np.vstack(X_imbalanced)
    y_imbalanced = np.array(y_imbalanced)

    return X_imbalanced, y_imbalanced

### for each ratio implement three ideas and evaluate them

In [8]:
for ratio in imbalance_ratios:
    
    # Create imbalanced training set
    X_train_imbalanced, y_train_imbalanced = create_imbalanced_dataset(X_train, y_train, ratio)

    # Train and evaluate logistic regression on imbalanced dataset
    clf_imbalanced = LogisticRegression(max_iter=1000, multi_class='auto', solver='lbfgs', random_state=42)
    clf_imbalanced.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_imbalanced = clf_imbalanced.predict(X_test)
    
    # Apply RandomOverSampler to balance the dataset by oversampling the minority classes
    random_over_sampler = RandomOverSampler(sampling_strategy='minority', random_state=42)
    X_train_balanced_with_ros, y_train_balanced_with_ros = random_over_sampler.fit_resample(X_train_imbalanced, y_train_imbalanced)

    # Train and evaluate logistic regression on balanced dataset (with random over sampling)
    model = LogisticRegression(max_iter=1000, multi_class='auto', solver='lbfgs', random_state=42)
    model.fit(X_train_balanced_with_ros, y_train_balanced_with_ros)
    y_pred_balanced_with_ros = model.predict(X_test)

    # Create and train the logistic regression model with class weighting
    model_with_class_weighting = LogisticRegression(class_weight='balanced', random_state=42, multi_class='auto', solver='lbfgs')
    model_with_class_weighting.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_class_weighting = model_with_class_weighting.predict(X_test)

    # Train and evaluate XGBClassifier with logistic regression as the base learner
    xgb_clf = XGBClassifier(booster='gblinear', objective='multi:softmax', num_class=len(np.unique(y)), n_estimators=100, learning_rate=0.1, random_state=42)
    xgb_clf.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_xgb = xgb_clf.predict(X_test)

    # Calculate performance metrics
    metrics_imbalanced = [accuracy_score(y_test, y_pred_imbalanced), f1_score(y_test, y_pred_imbalanced, average='weighted')]
    metrics_balanced_with_ros = [accuracy_score(y_test, y_pred_balanced_with_ros), f1_score(y_test, y_pred_balanced_with_ros, average='weighted')]
    metrics_balanced_with_class_weighting = [accuracy_score(y_test, y_pred_balanced_with_class_weighting), f1_score(y_test, y_pred_balanced_with_class_weighting, average='weighted')]
    metrics_balanced_with_xgb = [accuracy_score(y_test, y_pred_balanced_with_xgb), f1_score(y_test, y_pred_balanced_with_xgb, average='weighted')]

    print(f"Imbalance ratio: {ratio}")
    print("Imbalanced dataset metrics: Accuracy: {:.4f}, F1-score: {:.4f}".format(*metrics_imbalanced))
    print("Balanced dataset with random over sampling metrics: Accuracy: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_ros))
    print("Balanced dataset with class weighting metrics: Accuracy: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_class_weighting))
    print("Balanced dataset with xgb metrics: Accuracy: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_xgb))
    print("\n")

Imbalance ratio: (90, 5, 5)
Imbalanced dataset metrics: Accuracy: 0.9333, F1-score: 0.9319
Balanced dataset with random over sampling metrics: Accuracy: 0.9000, F1-score: 0.9003
Balanced dataset with class weighting metrics: Accuracy: 0.9667, F1-score: 0.9664
Balanced dataset with xgb metrics: Accuracy: 0.9333, F1-score: 0.9333


Imbalance ratio: (85, 10, 5)
Imbalanced dataset metrics: Accuracy: 0.9667, F1-score: 0.9668
Balanced dataset with random over sampling metrics: Accuracy: 0.9333, F1-score: 0.9319
Balanced dataset with class weighting metrics: Accuracy: 1.0000, F1-score: 1.0000
Balanced dataset with xgb metrics: Accuracy: 1.0000, F1-score: 1.0000


Imbalance ratio: (80, 15, 5)
Imbalanced dataset metrics: Accuracy: 0.9333, F1-score: 0.9333
Balanced dataset with random over sampling metrics: Accuracy: 0.9333, F1-score: 0.9319
Balanced dataset with class weighting metrics: Accuracy: 0.9667, F1-score: 0.9664
Balanced dataset with xgb metrics: Accuracy: 1.0000, F1-score: 1.0000


Im